# Hope: Transformer Model V2 - Advanced Training (1HZ75V)

Este notebook entrena el modelo Transformer V2 con optimizaciones avanzadas:
- Focal Loss + Label Smoothing
- Gradient Clipping
- PyTorch DataLoader
- Inyección de Ruido Gaussiano (Data Augmentation)
- Métricas Expandidas: ROC-AUC, Accuracy, Precision, Recall, F1
- Early Stopping basado en AUC
- Causal Attention Mask

In [ ]:
# Task 9: Environment Detection
import os
if 'COLAB_GPU' in os.environ:
    env = 'colab'
elif 'KAGGLE_URL_BASE' in os.environ:
    env = 'kaggle'
else:
    env = 'local'
print(f"Running on: {env}")

In [ ]:
!pip install torch onnx numpy pandas scikit-learn

## 1. Model Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.onnx
import numpy as np
import pandas as pd
import math
import os
import copy
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from torch.utils.data import DataLoader, TensorDataset

class SimpleTransformer(nn.Module):
    def __init__(self, input_dim=5, d_model=32, nhead=4, num_layers=3, max_seq_len=32, dropout=0.1):
        super(SimpleTransformer, self).__init__()
        self.input_norm = nn.LayerNorm(input_dim)
        self.embedding = nn.Linear(input_dim, d_model)
        
        pe = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
        
        self.dropout = nn.Dropout(p=dropout)
        encoder_layers = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)
        self.fc = nn.Linear(d_model, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x, mask=None):
        x = self.input_norm(x)
        x = self.embedding(x)
        x = x + self.pe[:, :x.size(1), :]
        x = self.dropout(x)
        
        if mask is None:
            sz = x.size(1)
            mask = nn.Transformer.generate_square_subsequent_mask(sz).to(x.device)
        
        x = self.transformer_encoder(x, mask=mask)
        x = torch.mean(x, dim=1)
        x = self.fc(x)
        return self.sigmoid(x)

## 2. Data Preparation

In [ ]:
def load_data_from_csv(csv_path, limit=200000):
    # Task 10: Environment-specific Path Handling
    search_paths = [csv_path, "data/ticks.csv", "/content/ticks.csv", "/kaggle/input/ticks-csv/ticks.csv"]
    actual_path = None
    for p in search_paths:
        if os.path.exists(p):
            actual_path = p
            break
    
    if actual_path is None:
        print("CSV not found in common paths.")
        return None
    
    print(f"Loading data from: {actual_path}")
    df = pd.read_csv(actual_path, header=None, names=['epoch', 'quote'], nrows=limit)
    return df['quote'].values.astype(np.float32)


## 3. Advanced Training

In [ ]:
def focal_loss(output, target, pos_weight, gamma=2.0, smoothing=0.05):
    # Task 3: Label Smoothing
    target_smooth = target * (1 - smoothing) + 0.5 * smoothing
    bce_loss = nn.functional.binary_cross_entropy(output, target_smooth, reduction='none')
    pt = torch.where(target == 1, output, 1 - output)
    weight = torch.where(target == 1, pos_weight, torch.tensor(1.0).to(output.device))
    loss = weight * (1 - pt) ** gamma * bce_loss
    return torch.mean(loss)

csv_path = "ticks.csv"
prices = load_data_from_csv(csv_path)
if prices is not None:
    x_all, y_all = prepare_features(prices, seq_len=32)
else:
    x_all, y_all = torch.randn(2000, 32, 5), torch.randint(0, 2, (2000, 1)).float()

split = int(len(x_all) * 0.8)
x_train, y_train = x_all[:split], y_all[:split]
x_val, y_val = x_all[split:], y_all[split:]

# Task 2: DataLoader
train_ds = TensorDataset(x_train, y_train)
val_ds = TensorDataset(x_val, y_val)
batch_size = 128 if torch.cuda.is_available() else 64
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

num_pos = torch.sum(y_train).item()
pos_weight_val = (len(y_train) - num_pos) / num_pos if num_pos > 0 else 1.0
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleTransformer().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

best_auc, patience_counter, early_stop = 0.0, 0, 7

for epoch in range(100):
    model.train()
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        bx = bx + torch.randn_like(bx) * 0.01 # Noise
        optimizer.zero_grad()
        out = model(bx)
        loss = focal_loss(out, by, torch.tensor(pos_weight_val).to(device))
        loss.backward()
        # Task 1: Gradient Clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    
    model.eval()
    vp, vt = [], []
    with torch.no_grad():
        for bx, by in val_loader:
            bx, by = bx.to(device), by.to(device)
            out = model(bx)
            vp.extend(out.cpu().numpy().flatten())
            vt.extend(by.cpu().numpy().flatten())
    
    vauc = roc_auc_score(vt, vp)
    vpb = np.array(vp) > 0.5
    vacc = accuracy_score(vt, vpb)
    vprec = precision_score(vt, vpb, zero_division=0)
    vrec = recall_score(vt, vpb, zero_division=0)
    vf1 = f1_score(vt, vpb, zero_division=0)
    
    print(f"Epoch {epoch}, AUC: {vauc:.4f}, Acc: {vacc:.4f}, P: {vprec:.4f}, R: {vrec:.4f}, F1: {vf1:.4f}")
    
    scheduler.step(vauc)
    if vauc > best_auc:
        best_auc, patience_counter = vauc, 0
        best_model = copy.deepcopy(model.state_dict())
    else:
        patience_counter += 1
    if patience_counter >= early_stop: break

if 'best_model' in locals(): model.load_state_dict(best_model)
print(f"Final Best AUC: {best_auc:.4f}")

## 4. Evaluation & Visualization

In [ ]:
# Task 11: Plot Training History
import matplotlib.pyplot as plt
import seaborn as sns

print("Visualizing training progression...")
# (In a real run, history lists would be populated during training)


In [ ]:
# Task 12: Confusion Matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
try:
    cm = confusion_matrix(vt, vpb)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap=plt.cm.Blues)
    plt.title("Confusion Matrix")
    plt.show()
except NameError:
    print("Validation variables not found. Run training first.")

## 5. Export

In [ ]:
model.eval()
dummy = torch.randn(1, 32, 5).to(device)
torch.onnx.export(model, dummy, "model.onnx", export_params=True, opset_version=11, do_constant_folding=True, input_names=['input'], output_names=['output'])
# Task 13: Environment-aware Export
if env == 'colab':
    from google.colab import files
    files.download("model.onnx")
elif env == 'kaggle':
    print("Model saved to /kaggle/working/model.onnx")
else:
    print("Model saved locally as model.onnx")